## Imports


In [1]:
import pandas as pd
import numpy as np
import ast

from shapely import wkb
from math import radians, sin, cos, sqrt, atan2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

## Load Data


In [2]:
train = pd.read_csv("/kaggle/input/competitions/ai-cup-2026-performance/train.csv")
test = pd.read_csv("/kaggle/input/competitions/ai-cup-2026-performance/test.csv")

train = train.set_index("track_id")
test = test.set_index("track_id")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (2601, 15)
Test: (1872, 8)


## Time Features


In [3]:
train["timestamp_start_radar_utc"] = pd.to_datetime(train["timestamp_start_radar_utc"])
test["timestamp_start_radar_utc"] = pd.to_datetime(test["timestamp_start_radar_utc"])

train["month"] = train["timestamp_start_radar_utc"].dt.month
train["hour"] = train["timestamp_start_radar_utc"].dt.hour

test["month"] = test["timestamp_start_radar_utc"].dt.month
test["hour"] = test["timestamp_start_radar_utc"].dt.hour

## Haversine Distance


In [4]:
def haversine(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

## Trajectory Features


In [5]:
def extract_features(row):
    geom = wkb.loads(bytes.fromhex(row["trajectory"]))
    coords = np.array(list(geom.coords))

    lon = coords[:, 0]
    lat = coords[:, 1]
    alt = coords[:, 2]
    rcs = coords[:, 3]

    t = np.array(ast.literal_eval(row["trajectory_time"]))

    feats = {}

    feats["traj_len"] = len(coords)
    feats["alt_mean"] = alt.mean()
    feats["alt_std"] = alt.std()
    feats["rcs_mean"] = rcs.mean()
    feats["rcs_std"] = rcs.std()

    dists = []
    speeds = []
    vecs = []

    for i in range(len(coords) - 1):
        d = haversine(lon[i], lat[i], lon[i + 1], lat[i + 1])
        dt = t[i + 1] - t[i]

        if dt <= 0:
            continue

        speed = d / dt

        dists.append(d)
        speeds.append(speed)

        dx = lon[i + 1] - lon[i]
        dy = lat[i + 1] - lat[i]

        vecs.append([dx, dy])

    dists = np.array(dists)
    speeds = np.array(speeds)

    feats["path_length"] = dists.sum()
    feats["step_mean"] = dists.mean()
    feats["step_std"] = dists.std()
    feats["speed_mean"] = speeds.mean()
    feats["speed_std"] = speeds.std()
    feats["speed_cv"] = feats["speed_std"] / (feats["speed_mean"] + 1e-6)

    alt_diff = np.diff(alt)

    feats["alt_climb_mean"] = alt_diff.mean()
    feats["alt_climb_std"] = alt_diff.std()

    speed_diff = np.diff(speeds)

    feats["acc_mean"] = speed_diff.mean()
    feats["acc_std"] = speed_diff.std()

    vecs = np.array(vecs)

    turns = []
    curvatures = []

    for i in range(len(vecs) - 1):
        v1 = vecs[i]
        v2 = vecs[i + 1]

        dot = np.dot(v1, v2)
        norm = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6

        angle = np.arccos(np.clip(dot / norm, -1, 1))
        turns.append(angle)

        cross = abs(v1[0] * v2[1] - v1[1] * v2[0])
        curvatures.append(cross / norm)

    turns = np.array(turns)
    curvatures = np.array(curvatures)

    feats["turn_mean"] = turns.mean()
    feats["turn_std"] = turns.std()

    feats["curvature_mean"] = curvatures.mean()
    feats["curvature_std"] = curvatures.std()

    start = coords[0]
    end = coords[-1]

    displacement = haversine(start[0], start[1], end[0], end[1])

    feats["displacement"] = displacement
    feats["straightness"] = displacement / (feats["path_length"] + 1e-6)
    feats["efficiency"] = displacement / (feats["path_length"] + 1e-6)
    feats["log_path_length"] = np.log1p(feats["path_length"])

    return pd.Series(feats)

In [6]:
print("Extracting trajectory features...")

train_feats = train.apply(extract_features, axis=1)
test_feats = test.apply(extract_features, axis=1)

train = pd.concat([train, train_feats], axis=1)
test = pd.concat([test, test_feats], axis=1)

Extracting trajectory features...


## Feature List


In [7]:
features = [
    "airspeed",
    "min_z",
    "max_z",
    "radar_bird_size",
    "month",
    "hour",
    "traj_len",
    "alt_mean",
    "alt_std",
    "alt_climb_mean",
    "alt_climb_std",
    "rcs_mean",
    "rcs_std",
    "path_length",
    "log_path_length",
    "step_mean",
    "step_std",
    "speed_mean",
    "speed_std",
    "speed_cv",
    "acc_mean",
    "acc_std",
    "turn_mean",
    "turn_std",
    "curvature_mean",
    "curvature_std",
    "displacement",
    "straightness",
    "efficiency",
]

X = train[features].copy()
X_test = test[features].copy()

y = train["bird_group"]

## Encoding


In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes = list(le.classes_)

In [9]:
X["radar_bird_size"] = X["radar_bird_size"].astype("category")
X_test["radar_bird_size"] = X_test["radar_bird_size"].astype("category")

## Storage


In [10]:
n_classes = len(classes)

oof_cat = np.zeros((len(X), n_classes))
oof_lgb = np.zeros((len(X), n_classes))
oof_xgb = np.zeros((len(X), n_classes))

test_cat = np.zeros((len(X_test), n_classes))
test_lgb = np.zeros((len(X_test), n_classes))
test_xgb = np.zeros((len(X_test), n_classes))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Train Models


In [11]:
for fold, (tr, val) in enumerate(skf.split(X, y_encoded)):
    print("\n>>>>> Fold", fold + 1)

    y_val = y_encoded[val]
    y_val_onehot = pd.get_dummies(y_val).reindex(columns=range(n_classes), fill_value=0).values

    # CatBoost
    cat = CatBoostClassifier(
        task_type="GPU",
        iterations=4000,
        depth=8,
        learning_rate=0.02,
        loss_function="MultiClass",
        verbose=0,
    )

    cat.fit(
        X.iloc[tr],
        y_encoded[tr],
        cat_features=["radar_bird_size"],
        eval_set=(X.iloc[val], y_encoded[val]),
        use_best_model=True,
    )

    cat_pred = cat.predict_proba(X.iloc[val])

    oof_cat[val] = cat_pred
    test_cat += cat.predict_proba(X_test) / 5

    cat_score = average_precision_score(y_val_onehot, cat_pred, average="macro")
    print("CatBoost mAP:", round(cat_score, 6))

    # LightGBM
    lgb = LGBMClassifier(
        device="gpu",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=64,
        verbose=-1
    )

    lgb.fit(X.iloc[tr], y_encoded[tr], categorical_feature=["radar_bird_size"])

    lgb_pred = lgb.predict_proba(X.iloc[val])

    oof_lgb[val] = lgb_pred
    test_lgb += lgb.predict_proba(X_test) / 5

    lgb_score = average_precision_score(y_val_onehot, lgb_pred, average="macro")
    print("LightGBM mAP:", round(lgb_score, 6))

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=2500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda",
        enable_categorical=True,
        eval_metric="mlogloss",
    )

    xgb.fit(X.iloc[tr], y_encoded[tr])

    xgb_pred = xgb.predict_proba(X.iloc[val])

    oof_xgb[val] = xgb_pred
    test_xgb += xgb.predict_proba(X_test) / 5

    xgb_score = average_precision_score(y_val_onehot, xgb_pred, average="macro")
    print("XGBoost mAP:", round(xgb_score, 6))

    # Fold Ensemble Score
    ensemble_pred = 0.65 * cat_pred + 0.20 * lgb_pred + 0.15 * xgb_pred

    ensemble_score = average_precision_score(y_val_onehot, ensemble_pred, average="macro")
    print("Ensemble mAP:", round(ensemble_score, 6))


>>>>> Fold 1
CatBoost mAP: 0.687691


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LightGBM mAP: 0.606455


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [14:12:27] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost mAP: 0.648376
Ensemble mAP: 0.676288

>>>>> Fold 2
CatBoost mAP: 0.662024
LightGBM mAP: 0.61321
XGBoost mAP: 0.640743
Ensemble mAP: 0.663368

>>>>> Fold 3
CatBoost mAP: 0.6953
LightGBM mAP: 0.646663
XGBoost mAP: 0.680213
Ensemble mAP: 0.690202

>>>>> Fold 4
CatBoost mAP: 0.666352
LightGBM mAP: 0.608719
XGBoost mAP: 0.620255
Ensemble mAP: 0.675652

>>>>> Fold 5
CatBoost mAP: 0.704708
LightGBM mAP: 0.652549
XGBoost mAP: 0.680445
Ensemble mAP: 0.697604


## CV Score


In [12]:
y_true = pd.get_dummies(y_encoded).values
final_oof = 0.65 * oof_cat + 0.20 * oof_lgb + 0.15 * oof_xgb
score = average_precision_score(y_true, final_oof, average="macro")
print("CV mAP:", score)

CV mAP: 0.6660682216320017


## Submission


In [13]:
sub = pd.read_csv("/kaggle/input/competitions/ai-cup-2026-performance/sample_submission.csv")
final_test = 0.65 * test_cat + 0.20 * test_lgb + 0.15 * test_xgb
sub[classes] = final_test
sub.to_csv("submission.csv", index=False)
print("Submission saved!")

Submission saved!
